In [2]:
import torch
import gc
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import create_pan_cameras, decode_latent_images, gif_widget
import json

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
xm = load_model('transmitter', device=device)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()
gc.collect()

In [4]:
model = load_model('text300M', device=device)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()
gc.collect()

In [5]:
diffusion = diffusion_from_config(load_config('diffusion'))
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()
gc.collect()

In [11]:
with open("/content/shap-e_GET3D/shap_e/examples/config.json", "r") as f:
    config = json.load(f)

In [12]:
batch_size = int(config.get("batch_size", 4))
guidance_scale = float(config.get("guidance_scale", 1.0))
prompt = str(config.get("prompt", "a shark"))
render_mode = str(config.get("render_mode", "nerf")).lower()
size = max(32, int(config.get("size", 32)))

In [13]:
if render_mode not in ['nerf', 'stf']:
    render_mode = 'nerf'

In [6]:
latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(texts=[prompt] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    device=device,
    s_churn=0,
)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()
gc.collect()

In [ ]:
cameras = create_pan_cameras(size, device)
for i, latent in enumerate(latents):
    images = decode_latent_images(xm, latent, cameras, rendering_mode=render_mode)
    display(gif_widget(images))

In [ ]:
# Example of saving the latents as meshes.
from shap_e.util.notebooks import decode_latent_mesh

for i, latent in enumerate(latents):
    t = decode_latent_mesh(xm, latent).tri_mesh()
    with open(f'example_mesh_{i}.ply', 'wb') as f:
        t.write_ply(f)
    with open(f'example_mesh_{i}.obj', 'w') as f:
        t.write_obj(f)

In [ ]:
project_name = 'output'
blend_file_path = '/content/shap-e_GET3D/example_mesh_3.obj'

# Change directory
%cd /content

import os

# Create required directories
!mkdir -p /content/$project_name/rendered
!mkdir -p /content/$project_name/blend

# Copy the .obj file to the blend directory
!cp $blend_file_path /content/$project_name/blend/

blend_file = os.path.basename(blend_file_path)
blend_file_name = os.path.splitext(blend_file)[0]  # Extract file name without extension

# Blender download and setup
blender_url = "https://ftp.nluug.nl/pub/graphics/blender/release/Blender3.3/blender-3.3.8-linux-x64.tar.xz"
base_url = os.path.basename(blender_url)
blender_version = 'blender-3.3.8'

!mkdir $blender_version
!wget -nc $blender_url
!tar -xkf $base_url -C ./$blender_version --strip-components=1
!rm $base_url

# GPU and CPU configuration script
gpu_enabled = True
cpu_enabled = False

gpu_script = f"""
import re
import bpy
scene = bpy.context.scene
scene.cycles.device = 'GPU'
prefs = bpy.context.preferences
prefs.addons['cycles'].preferences.get_devices()
cprefs = prefs.addons['cycles'].preferences
print(cprefs)
for compute_device_type in ('CUDA', 'OPENCL', 'NONE'):
    try:
        cprefs.compute_device_type = compute_device_type
        print('Device found:', compute_device_type)
        break
    except TypeError:
        pass
for device in cprefs.devices:
    if not re.match('intel', device.name, re.I):
        print('Activating', device)
        device.use = {gpu_enabled}
    else:
        device.use = {cpu_enabled}
"""
with open('setgpu.py', 'w') as f:
    f.write(gpu_script)

# Ensure the GPU script is in Blender's directory
!cp /content/setgpu.py /content/$blender_version/

# Convert OBJ to BLEND
conversion_script = f"""
import bpy

# Delete all objects except for cameras and lights
for obj in bpy.data.objects:
    if obj.type not in {'CAMERA', 'LIGHT'}:
        bpy.data.objects.remove(obj, do_unlink=True)

# Load the OBJ file
bpy.ops.import_scene.obj(filepath='/content/{project_name}/blend/{blend_file}')

# Save as BLEND file
bpy.ops.wm.save_as_mainfile(filepath='/content/{project_name}/blend/{blend_file_name}.blend')
"""


with open('convert_obj_to_blend.py', 'w') as f:
    f.write(conversion_script)

# Run Blender to convert OBJ to BLEND
!./$blender_version/blender -b -P convert_obj_to_blend.py

# Render the .blend file
start_frame = 1
output_path = f'/content/{project_name}/rendered/{project_name}-###'

!./$blender_version/blender -b "/content/$project_name/blend/{blend_file_name}.blend" -P setgpu.py -E CYCLES -o "$output_path" -noaudio -f $start_frame